# KKBox — Phase 5: Root Cause Analysis (RCA) Pareto

**Goal:** Classify *why* churned/leaked users left and rank leakage drivers by revenue impact.

**RCA Logic (strict sequential CASE WHEN on the last transaction row):**
1. `is_cancel = 1`            → **Voluntary Churn** (explicit cancellation)
2. `is_payment_failed = 1`    → **Involuntary Payment Failure** (card declined)
3. `amount_paid < plan_price` → **Discount Leakage** (pricing failure)
4. `is_auto_renew = 0`        → **Passive Expiry** (manual plan lapsed naturally)
5. ELSE                       → **Silent Abandonment** (no signal, just gone)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import psycopg2
from IPython.display import display, Markdown

DB = {
    "host":     "localhost",
    "port":     5432,
    "dbname":   "kkbox",
    "user":     "postgres",
    "password": "password",
}

def get_conn():
    return psycopg2.connect(**DB)

conn = get_conn()
print("DB connection OK")

DB connection OK


## Step 1 — Sanity Check: Dataset Date Range

The data is historical (not live).  
We define **"churned"** as: `MAX(expire_date)` falls *before* the dataset's last `transaction_date`,  
AND the user has **no transaction after their final expire_date** (i.e. they never renewed).

In [2]:
DATE_RANGE_SQL = """
SELECT
    MIN(transaction_date) AS min_txn_date,
    MAX(transaction_date) AS max_txn_date,
    MIN(expire_date)      AS min_expire_date,
    MAX(expire_date)      AS max_expire_date,
    COUNT(DISTINCT user_id) AS total_users
FROM transactions;
"""

conn = get_conn()
df_range = pd.read_sql(DATE_RANGE_SQL, conn)
conn.close()

display(df_range)

C:\Users\Vamsi Bhogaraju\AppData\Local\Temp\ipykernel_83840\1525717789.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_range = pd.read_sql(DATE_RANGE_SQL, conn)


,min_txn_date,max_txn_date,min_expire_date,max_expire_date,total_users
0,2015-01-01,2017-03-31,2016-04-19,2036-10-15,1197050


## Step 2 — The RCA Engine (Full SQL Pipeline)

Six sequential CTEs:
1. `user_last_expire` — MAX(expire_date) per user
2. `dataset_end` — observation window cutoff (MAX transaction_date)
3. `churned` — users whose final subscription has lapsed with no renewal
4. `last_txn_ranked` — row_number to isolate the single last transaction per churned user
5. `rca` — strict sequential CASE WHEN classification
6. `pareto` — group + aggregate with window function for share %

In [3]:
RCA_SQL = """
WITH

/* ────────────────────────────────────────────────────────────
   STEP 1 — Last expire date per user
   ──────────────────────────────────────────────────────────── */
user_last_expire AS (
    SELECT
        user_id,
        MAX(expire_date) AS last_expire_date
    FROM transactions
    GROUP BY user_id
),

/* ────────────────────────────────────────────────────────────
   STEP 2 — Observation window: last transaction date in the dataset
   ──────────────────────────────────────────────────────────── */
dataset_end AS (
    SELECT MAX(transaction_date) AS end_date
    FROM transactions
),

/* ────────────────────────────────────────────────────────────
   STEP 3 — Churned / Leaked users
   Definition:
     - last_expire_date is BEFORE the dataset end date
       (subscription has expired within our observation window)
     - AND no transaction_date exists AFTER last_expire_date
       (they never renewed / came back)
   ──────────────────────────────────────────────────────────── */
churned AS (
    SELECT ule.user_id,
           ule.last_expire_date
    FROM   user_last_expire ule
    CROSS  JOIN dataset_end de
    WHERE  ule.last_expire_date < de.end_date
      AND  NOT EXISTS (
               SELECT 1
               FROM   transactions t
               WHERE  t.user_id          = ule.user_id
                 AND  t.transaction_date > ule.last_expire_date
           )
),

/* ────────────────────────────────────────────────────────────
   STEP 4 — Isolate the single LAST transaction row per churned user
   Tie-break: latest expire_date first (most recent plan renewal)
   ──────────────────────────────────────────────────────────── */
last_txn_ranked AS (
    SELECT
        t.user_id,
        t.transaction_date,
        t.expire_date,
        t.plan_price,
        t.amount_paid,
        t.is_cancel,
        t.is_payment_failed,
        t.is_auto_renew,
        ROW_NUMBER() OVER (
            PARTITION BY t.user_id
            ORDER BY t.transaction_date DESC,
                     t.expire_date      DESC
        ) AS rn
    FROM transactions t
    JOIN churned      c ON c.user_id = t.user_id
),

last_txn AS (
    SELECT * FROM last_txn_ranked WHERE rn = 1
),

/* ────────────────────────────────────────────────────────────
   STEP 5 — RCA Classification (strict sequential CASE WHEN)
   Priority order is intentional — each condition is mutually
   exclusive once the prior has been ruled out.
   ──────────────────────────────────────────────────────────── */
rca AS (
    SELECT
        user_id,
        plan_price,
        amount_paid,
        is_cancel,
        is_payment_failed,
        is_auto_renew,
        CASE
            WHEN is_cancel         = 1 THEN 'Voluntary Churn'
            WHEN is_payment_failed = 1 THEN 'Involuntary Payment Failure'
            WHEN amount_paid < plan_price  THEN 'Discount Leakage'
            WHEN is_auto_renew     = 0 THEN 'Passive Expiry'
            ELSE                            'Silent Abandonment'
        END AS leakage_category
    FROM last_txn
),

/* ────────────────────────────────────────────────────────────
   STEP 6 — Pareto Aggregation
   Window function computes each category's share of total
   revenue lost in one pass — no self-join needed.
   ──────────────────────────────────────────────────────────── */
pareto AS (
    SELECT
        leakage_category,
        COUNT(*)                                               AS user_count,
        SUM(plan_price)                                        AS total_revenue_lost,
        ROUND(
            SUM(plan_price) * 100.0
            / NULLIF(SUM(SUM(plan_price)) OVER (), 0)
        , 2)                                                   AS pareto_share_pct
    FROM rca
    GROUP BY leakage_category
)

SELECT
    leakage_category                     AS "Leakage_Category",
    user_count                           AS "User_Count",
    total_revenue_lost                   AS "Total_Revenue_Lost",
    pareto_share_pct                     AS "Pareto_Share_Pct"
FROM  pareto
ORDER BY total_revenue_lost DESC;
"""

conn = get_conn()
df_rca = pd.read_sql(RCA_SQL, conn)
conn.close()

print(f"Query complete. {len(df_rca)} leakage categories found.")
print(f"Total churned users covered: {df_rca['User_Count'].sum():,}")

C:\Users\Vamsi Bhogaraju\AppData\Local\Temp\ipykernel_83840\1756632317.py:123: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_rca = pd.read_sql(RCA_SQL, conn)


Query complete. 3 leakage categories found.
Total churned users covered: 8,645


## Step 3 — Pareto Summary Table

In [ ]:
# ── Compute cumulative share for the 80/20 Pareto line ──────────────────────
df_display = df_rca.copy()
df_display = df_display.sort_values("Total_Revenue_Lost", ascending=False).reset_index(drop=True)

df_display["Cumulative_Pct"] = df_display["Pareto_Share_Pct"].cumsum().round(2)

# ── Formatting for display ──────────────────────────────────────────────────
df_print = df_display.copy()
df_print["User_Count"]         = df_print["User_Count"].map("{:,}".format)
df_print["Total_Revenue_Lost"] = df_print["Total_Revenue_Lost"].map("${:,.0f}".format)
df_print["Pareto_Share_Pct"]   = df_print["Pareto_Share_Pct"].map("{:.2f}%".format)
df_print["Cumulative_Pct"]     = df_print["Cumulative_Pct"].map("{:.2f}%".format)

df_print.index = range(1, len(df_print) + 1)   # rank from 1
df_print.index.name = "Rank"

# ── Render Markdown table ───────────────────────────────────────────────────
md_table = df_print.to_markdown(tablefmt="github")
display(Markdown("### RCA Pareto: Revenue Leakage by Root Cause\n"))
display(Markdown(md_table))
print()
print(df_print.to_markdown(tablefmt="github"))   # plain-text fallback

## Step 4 — Pareto Bar + Cumulative Line Chart

In [ ]:
import matplotlib.pyplot as plt
import os

os.makedirs("./Data/outputs", exist_ok=True)

# 1. Format the raw DataFrame into a simple 2D List for the table drawer
table_data = []
for idx, row in df_display.iterrows():
    table_data.append([
        row['Leakage_Category'],
        f"{row['User_Count']:,}",
        f"${row['Total_Revenue_Lost']:,.0f}",
        f"{row['Pareto_Share_Pct']:.2f}%"
    ])

columns = ["Root Cause Category", "Users Lost", "Revenue Lost ($)", "Share of Leakage"]

# 2. Create a blank Matplotlib canvas
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.axis("off")

# 3. Draw the Table Component
table = ax.table(
    cellText=table_data,
    colLabels=columns,
    cellLoc="center",
    loc="center",
    edges="closed"
)

# 4. Polish the Table Aesthetics
table.auto_set_font_size(False)
table.set_fontsize(12)
table.scale(1.2, 2.2)

for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_facecolor("#1e293b")
        cell.set_text_props(color="white", weight="bold")
    else:
        cell.set_facecolor("#f8fafc" if row % 2 == 0 else "white")
        cell.set_edgecolor("#cbd5e1")
        if row == 1:
            cell.set_text_props(weight="bold", color="#1e293b")

plt.title("KKBox Revenue Leakage — Root Cause Table", fontsize=18, fontweight="bold", color="#0f172a", pad=20)
plt.savefig("./Data/outputs/rca_table.png", dpi=200, bbox_inches="tight", facecolor="white")
plt.show()

print("High-Res Table Image saved → ./Data/outputs/rca_table.png")

## Step 5 — Executive Interpretation

In [ ]:
top_category  = df_display.iloc[0]
top2_share    = df_display.iloc[:2]["Pareto_Share_Pct"].sum()
total_users   = df_display["User_Count"].sum()
total_revenue = df_display["Total_Revenue_Lost"].sum()

summary = f"""
## Executive Summary — RCA Pareto Findings

| Metric                          | Value                            |
|---------------------------------|----------------------------------|
| Total Churned Users Analysed    | {total_users:,}                  |
| Total Revenue at Risk ($)       | ${total_revenue:,.0f}             |
| #1 Leakage Driver               | **{top_category['Leakage_Category']}** ({top_category['Pareto_Share_Pct']:.2f}% of revenue lost) |
| Top-2 Categories Combined       | {top2_share:.2f}% of total revenue lost |

### Key Findings

1. **#1 Driver — {df_display.iloc[0]['Leakage_Category']}**  
   {df_display.iloc[0]['User_Count']:,} users → ${df_display.iloc[0]['Total_Revenue_Lost']:,.0f} ({df_display.iloc[0]['Pareto_Share_Pct']:.2f}%)  
   *Action: {"Investigate exit surveys & win-back campaigns" if 'Voluntary' in df_display.iloc[0]['Leakage_Category'] else "Improve payment retry logic & card updater service" if 'Payment' in df_display.iloc[0]['Leakage_Category'] else "Enforce minimum price floors; audit discount policy" if 'Discount' in df_display.iloc[0]['Leakage_Category'] else "Convert manual-pay users to auto-renew via UX nudges" if 'Passive' in df_display.iloc[0]['Leakage_Category'] else "Deploy re-engagement email/push before expiry"}*

2. **#2 Driver — {df_display.iloc[1]['Leakage_Category']}**  
   {df_display.iloc[1]['User_Count']:,} users → ${df_display.iloc[1]['Total_Revenue_Lost']:,.0f} ({df_display.iloc[1]['Pareto_Share_Pct']:.2f}%)  
   *Action: {"Investigate exit surveys & win-back campaigns" if 'Voluntary' in df_display.iloc[1]['Leakage_Category'] else "Improve payment retry logic & card updater service" if 'Payment' in df_display.iloc[1]['Leakage_Category'] else "Enforce minimum price floors; audit discount policy" if 'Discount' in df_display.iloc[1]['Leakage_Category'] else "Convert manual-pay users to auto-renew via UX nudges" if 'Passive' in df_display.iloc[1]['Leakage_Category'] else "Deploy re-engagement email/push before expiry"}*

> **Pareto Principle Check:** The top {(df_display['Cumulative_Pct'] <= 80).sum() + 1} categories account for ≥80% of total revenue leaked.
"""

display(Markdown(summary))